# Intent Classification Evaluation Notebook

This notebook allows you to evaluate your intent classification model without modifying any source code. It will:

1. Split your master CSV into train/val/test (70/15/15).
2. Overwrite the CSV with only the train set for training.
3. Run the original `train.py` script.
4. Evaluate the trained model on the test set (classification report, confusion matrix, error analysis).
5. Optionally restore the original CSV.

**All steps are performed externally, so your codebase remains unchanged.**

In [ ]:
# 1. Import libraries and split master data
import pandas as pd
from sklearn.model_selection import train_test_split

# Read master data (ensure hard cases are included)
df_master = pd.read_csv("intent_binary_dataset.csv").dropna().drop_duplicates(subset=['text'])

# Split into train (70%), val (15%), test (15%)
X = df_master['text'].tolist()
y = df_master['label'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# Overwrite CSV with only train set
df_train = pd.DataFrame({'text': X_train, 'label': y_train})
df_train.to_csv("intent_binary_dataset.csv", index=False)
print("Đã ghi đè 70% dữ liệu vào intent_binary_dataset.csv để chuẩn bị train!")

## Train the Model

Now, run the original `train.py` script. This will use the overwritten CSV (containing only the train set).

In [ ]:
# 2. Run the original training script (no code modification)
!python train.py
# Wait for the script to finish and save the model as usual.

## Evaluate on the Test Set

After training, evaluate the model using the 15% test set. This includes a classification report, confusion matrix, and error analysis.

In [ ]:
# 3. Evaluate the trained model on the test set
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix

# Load the trained model and encoder
model = SentenceTransformer("models/intent_encoder", device="cpu")
clf = joblib.load("models/intent_classifier.pkl")

# Encode test set and predict
y_pred = clf.predict(model.encode(X_test, show_progress_bar=True))

# Print classification report
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=["0 (No Recommend)", "1 (Recommend)"]))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Đoán 0", "Đoán 1"], yticklabels=["Thực 0", "Thực 1"])
plt.title('Confusion Matrix')
plt.show()

# Print misclassified examples
print("\n--- NHỮNG CÂU MODEL ĐOÁN SAI ---")
for text, true_l, pred_l in zip(X_test, y_test, y_pred):
    if true_l != pred_l:
        print(f"Thực tế {true_l} -> Đoán {pred_l} | Câu: {text}")

## (Optional) Restore the Full Master Dataset

After evaluation, you can restore the original CSV with all data for future use or archiving.

In [ ]:
# 4. Restore the full master dataset (optional)
df_master.to_csv("intent_binary_dataset.csv", index=False)
print("Đã khôi phục file intent_binary_dataset.csv về trạng thái 100%!")